# Commodity Price Forecasting

## Project Overview

Forecasts a **daily commodity price index** (USD) over a 14-day horizon. Synthetic data spans 2 years with mean-reverting behavior, supply/demand shocks, and mild seasonality.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily commodity prices, predict the next 14 days. Commodity price forecasts inform procurement strategies, hedging decisions, inflation expectations, and agricultural planning.

## Why This Project Matters

Commodity markets underpin the global economy. Raw material price movements affect manufacturing costs, consumer prices, trade balances, and geopolitical stability. Accurate short-term forecasts enable better hedging, inventory management, and risk mitigation.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily commodity price
- Mean-reverting around a slowly drifting equilibrium
- Supply/demand shock events
- Mild weekly pattern (trading days)
- Volatility clustering

| Column | Description |
|--------|-------------|
| `date` | Date |
| `price_usd` | Daily commodity price (USD per unit) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'price_usd'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

# Mean-reverting price (Ornstein-Uhlenbeck-like)
price = np.zeros(N_POINTS)
price[0] = 100
mu = 100 + 0.02 * t  # slowly drifting equilibrium
theta = 0.05  # mean reversion speed
sigma = 2.0

for i in range(1, N_POINTS):
    price[i] = price[i-1] + theta * (mu[i] - price[i-1]) + sigma * np.random.normal()

# Supply/demand shocks
for s in range(0, N_POINTS, 120):
    shock_day = min(s + np.random.randint(30, 90), N_POINTS - 1)
    shock = np.random.choice([-1, 1]) * np.random.uniform(5, 15)
    duration = min(np.random.randint(3, 10), N_POINTS - shock_day)
    for j in range(duration):
        price[shock_day + j] += shock * np.exp(-0.3 * j)

# Mild weekly pattern (lower on weekends)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow >= 5:
        price[i] -= 1.5

price_usd = np.maximum(price, 10).round(2)

df = pd.DataFrame({'date': dates, 'price_usd': price_usd})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,price_usd
0,2022-01-01,98.50
1,2022-01-02,99.49
2,2022-01-03,100.67
3,2022-01-04,101.94
4,2022-01-05,104.89


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('price_usd Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("price_usd Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

price_usd Statistics:
count    730.000000
mean     106.054507
std        7.005831
min       84.470000
25%      101.700000
50%      106.045000
75%      110.880000
max      126.700000
Name: price_usd, dtype: float64

CV: 0.066
Skewness: -0.145


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:        2.3 | RMSE:        3.0 | MAPE:  2.09%
Seasonal Naive (7)             | MAE:        3.6 | RMSE:        4.0 | MAPE:  3.29%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                               Adjusted R-Squared    R-Squared        RMSE  Time Taken
Model                                                                                 
KernelRidge                          26893.445539 -2067.649657  106.036383    0.047745
GaussianProcessRegressor              3342.924158  -256.071089   37.379893    0.038385
MLPRegressor                           268.938158   -19.610628   10.584176    0.244651
QuantileRegressor                       48.407527    -2.646733    4.452085    0.045231
DummyRegressor                          46.027214    -2.463632    4.338877    0.006737
PassiveAggressiveRegressor              41.527610    -2.117508    4.116378    0.006470
ExtraTreeRegressor                      32.413687    -1.416437    3.624092    0.007809
BaggingRegressor                        20.066753    -0.466673    2.823437    0.050249
HistGradientBoostingRegressor           19.454838    -0.419603    2.777761    0.136888
LGBMRegressor         

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: lgbm
FLAML (lgbm)                   | MAE:        2.0 | RMSE:        2.4 | MAPE:  1.79%
FLAML Test (lgbm)              | MAE:        1.8 | RMSE:        2.6 | MAPE:  1.62%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:        2.4 | RMSE:        3.2 | MAPE:  2.16%
SF AutoETS                     | MAE:        2.2 | RMSE:        2.8 | MAPE:  1.97%
SF SeasonalNaive               | MAE:        2.4 | RMSE:        3.3 | MAPE:  2.20%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
             model      MAE     RMSE     MAPE
 FLAML Test (lgbm) 1.751690 2.572497 1.619113
      FLAML (lgbm) 1.951720 2.391817 1.791972
        SF AutoETS 2.154555 2.791706 1.967555
Naive (Last Value) 2.279286 3.028770 2.088357
      SF AutoARIMA 2.378443 3.233244 2.162140
  SF SeasonalNaive 2.426429 3.294098 2.199650
Seasonal Naive (7) 3.624286 3.983439 3.288555

Top 3:
            model      MAE     RMSE     MAPE
FLAML Test (lgbm) 1.751690 2.572497 1.619113
     FLAML (lgbm) 1.951720 2.391817 1.791972
       SF AutoETS 2.154555 2.791706 1.967555


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -0.46, Std: 2.53


## Interpretation and Business Insight

- Commodity prices exhibit mean-reverting behavior around a slowly rising equilibrium
- Supply/demand shocks create temporary price spikes or drops
- Shock effects decay exponentially over several days
- Weekday vs weekend patterns reflect trading activity
- Long-term drift captures inflation/demand growth

## Limitations

1. Synthetic — real commodity prices depend on global supply/demand, geopolitics, currency
2. No fundamental factors (inventory levels, production data, trade flows)
3. No cross-commodity correlations (oil affects transportation costs)
4. Simplified volatility — real commodities have fat-tailed returns
5. No futures curve structure

## How to Improve This Project

1. Add macroeconomic indicators (GDP, inflation, exchange rates)
2. Include supply-side data (production, inventory, shipping)
3. Model returns rather than levels for stationarity
4. Use GARCH for volatility forecasting
5. Incorporate sentiment from news/social media

## Production Considerations

- Daily price forecasts for procurement optimization
- Hedging strategy support and risk metrics
- Integration with ERP/supply chain systems
- Automated trading signal generation (with extreme caution)

## Common Mistakes

1. Treating commodity prices as stationary
2. Ignoring mean-reversion when using trend-following models
3. Not modeling returns — levels have unit root issues
4. Overfitting to noise in daily price changes
5. Using point forecasts without prediction intervals

## Mini Challenge / Exercises

1. Convert prices to daily returns and compare stationarity tests
2. Detect supply/demand shock events from price jumps
3. Build a GARCH model for volatility forecasting
4. Compare directional accuracy vs RMSE as evaluation metrics

## Final Summary / Key Takeaways

1. Commodity prices mean-revert around a slowly shifting equilibrium
2. Supply/demand shocks are the hardest-to-forecast component
3. Working with returns improves stationarity and model performance
4. Prediction intervals matter more than point forecasts in finance
5. Fundamental data (inventories, production) is critical for real deployment

In [18]:
import json
metrics = {
    'project': 'Commodity Price Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Commodity Price Forecasting — Complete ===")

Metrics saved.

=== Commodity Price Forecasting — Complete ===
